# Hyperparameter tuning and the final model selection

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.feature_selection import VarianceThreshold
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor, RandomForestRegressor


In [2]:

df = pd.read_csv("..\Data\CSVs\Dopamine_D2_receptor_04_bioactivity_data_model.csv")
X = df.drop(columns=['pIC50'])
y = df['pIC50']


<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\HP\AppData\Local\Temp\ipykernel_9592\1241736746.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_csv("..\Data\CSVs\Dopamine_D2_receptor_04_bioactivity_data_model.csv")


In [3]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [4]:
# This applies the 0.1 variance threshold automatically before passing data to the model
pipelines = {
    'HistGB': Pipeline([
        ('vt', VarianceThreshold(threshold=0.1)),
        ('model', HistGradientBoostingRegressor(random_state=42))
    ]),
    'GBR': Pipeline([
        ('vt', VarianceThreshold(threshold=0.1)),
        ('model', GradientBoostingRegressor(random_state=42))
    ]),
    'RF': Pipeline([
        ('vt', VarianceThreshold(threshold=0.1)),
        ('model', RandomForestRegressor(random_state=42))
    ])
}


In [5]:

# We use 'model__' in front of the parameter names because they are inside a Pipeline
grids = {
    'HistGB': {
        'model__learning_rate': [0.05, 0.1, 0.2],    
        'model__max_iter': [100, 200, 300],         
        'model__max_depth': [3, 5, None]             
    },
    'GBR': {
        'model__learning_rate': [0.05, 0.1, 0.2],
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [3, 5, 7]
    },
    'RF': {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 10, 20],
        'model__min_samples_split': [2, 5, 10]      
    }
}


In [6]:
%%time
best_models = {}

for name in pipelines.keys():
    print(f"--- Training and Tuning {name} ---")
    
    grid_search = GridSearchCV(
        estimator=pipelines[name],
        param_grid=grids[name],
        cv=5,                 
        scoring='r2'
    )
    
    grid_search.fit(X_train, y_train)
    
    # Save the best version of the model
    best_models[name] = grid_search.best_estimator_
    
    print(f"Best Parameters: {grid_search.best_params_}")
    print(f"Best CV R2 Score: {grid_search.best_score_:.4f}\n")


--- Training and Tuning HistGB ---
Best Parameters: {'model__learning_rate': 0.2, 'model__max_depth': 5, 'model__max_iter': 200}
Best CV R2 Score: 0.6537

--- Training and Tuning GBR ---
Best Parameters: {'model__learning_rate': 0.1, 'model__max_depth': 3, 'model__n_estimators': 300}
Best CV R2 Score: 0.5895

--- Training and Tuning RF ---
Best Parameters: {'model__max_depth': 20, 'model__min_samples_split': 2, 'model__n_estimators': 200}
Best CV R2 Score: 0.5984

CPU times: total: 13min 26s
Wall time: 9min 11s


In [7]:

for name, model in best_models.items():

    y_pred = model.predict(X_test)
    
    test_r2 = r2_score(y_test, y_pred)
    test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    print(f"{name}:")
    print(f"  Test R-Squared : {test_r2:.4f}")
    print(f"  Test RMSE      : {test_rmse:.4f}\n")

HistGB:
  Test R-Squared : 0.7330
  Test RMSE      : 0.8278

GBR:
  Test R-Squared : 0.7475
  Test RMSE      : 0.8050

RF:
  Test R-Squared : 0.6262
  Test RMSE      : 0.9795



The best model is Gradient Boost Regrossor  which we will use as a final model

In [8]:
final_gbr_pipeline = best_models['GBR']
best_params = final_gbr_pipeline.get_params()

In [10]:
best_params.keys()

dict_keys(['memory', 'steps', 'transform_input', 'verbose', 'vt', 'model', 'vt__threshold', 'model__alpha', 'model__ccp_alpha', 'model__criterion', 'model__init', 'model__learning_rate', 'model__loss', 'model__max_depth', 'model__max_features', 'model__max_leaf_nodes', 'model__min_impurity_decrease', 'model__min_samples_leaf', 'model__min_samples_split', 'model__min_weight_fraction_leaf', 'model__n_estimators', 'model__n_iter_no_change', 'model__random_state', 'model__subsample', 'model__tol', 'model__validation_fraction', 'model__verbose', 'model__warm_start'])

In [9]:
for param_name in sorted(best_params.keys()):
    if 'model__' in param_name or 'vt__' in param_name:
        print(f"{param_name}: {best_params[param_name]}")

model__alpha: 0.9
model__ccp_alpha: 0.0
model__criterion: friedman_mse
model__init: None
model__learning_rate: 0.1
model__loss: squared_error
model__max_depth: 3
model__max_features: None
model__max_leaf_nodes: None
model__min_impurity_decrease: 0.0
model__min_samples_leaf: 1
model__min_samples_split: 2
model__min_weight_fraction_leaf: 0.0
model__n_estimators: 300
model__n_iter_no_change: None
model__random_state: 42
model__subsample: 1.0
model__tol: 0.0001
model__validation_fraction: 0.1
model__verbose: 0
model__warm_start: False
vt__threshold: 0.1
